#  Data Cleaning & Wrangling

**Goal:** turn the raw extraction into one trustworthy, analysis-ready table.

- **Input:**  `data/products_europe.csv`  (your ~70k balanced sample)
- **Output:** `data/products_clean.csv`  (used by every later notebook)

For more info: check `sample_data.py`

In [1]:
## 1. Setup
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

DATA_DIR = Path("../data") if Path("../data").exists() else Path("data")
RAW_PATH   = DATA_DIR / "products_europe.csv"
CLEAN_PATH = DATA_DIR / "products_clean.csv"
print("Reading from:", RAW_PATH.resolve())

Reading from: /Users/kanakyadav/Documents/GitHub/capstone/data/products_europe.csv


In [2]:
## 2. Load the raw data and take a first look
df = pd.read_csv(RAW_PATH, dtype={"code": "string"}, parse_dates=["last_modified"])
print("Shape:", df.shape)
df.head()
df.info()


Shape: (70000, 20)
<class 'pandas.DataFrame'>
RangeIndex: 70000 entries, 0 to 69999
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   code                70000 non-null  string 
 1   product_name        70000 non-null  str    
 2   brands              50794 non-null  str    
 3   country             70000 non-null  str    
 4   country_tag         70000 non-null  str    
 5   nutriscore_grade    69998 non-null  str    
 6   nutriscore_score    23675 non-null  float64
 7   nova_group          18873 non-null  float64
 8   eco_grade           67662 non-null  str    
 9   additives_n         20684 non-null  float64
 10  energy_kcal_100g    54423 non-null  float64
 11  sugars_100g         51871 non-null  float64
 12  fat_100g            53961 non-null  float64
 13  saturated_fat_100g  51200 non-null  float64
 14  salt_100g           47078 non-null  float64
 15  sodium_100g         47078 non-null  float64
 

In [3]:
# Numeric summary — a quick way to spot impossible values (e.g. max sugar > 100)
df.describe(include="number").T

,count,mean,std,min,25%,50%,75%,max
nutriscore_score,23675.0,10.241436,10.002610,-17.000000,2.000,9.000000,18.000000,45.000000
nova_group,18873.0,3.307635,1.048320,1.000000,3.000,4.000000,4.000000,4.000000
additives_n,20684.0,1.709582,2.470583,0.000000,0.000,1.000000,3.000000,25.000000
energy_kcal_100g,54423.0,269.096712,275.382142,-113.860001,107.000,248.000000,388.000000,31500.000000
sugars_100g,51871.0,11.316469,17.821309,0.000000,0.600,3.111111,12.400000,127.272728
fat_100g,53961.0,13.649153,19.604562,0.000000,1.200,7.000000,20.700001,1600.000000
saturated_fat_100g,51200.0,4.990961,8.594673,0.000000,0.300,1.800000,6.700000,829.099976
salt_100g,47078.0,1.437242,12.491356,0.000000,0.080,0.520000,1.400000,1890.000000
sodium_100g,47078.0,0.590370,5.810742,0.000000,0.032,0.204000,0.560000,756.000000
proteins_100g,54051.0,10.137944,19.506884,0.000000,2.100,6.800000,14.000000,3500.000000


In [4]:
## 3. Missing-value audit (look before you touch)

# Measure what's missing first. These numbers are worth quoting in your write-up.
missing = pd.DataFrame({
    "missing":   df.isna().sum(),
    "missing_%": (df.isna().mean() * 100).round(1),
}).sort_values("missing_%", ascending=False)
missing



,missing,missing_%
nova_group,51127,73.0
additives_n,49316,70.5
nutriscore_score,46325,66.2
fiber_100g,43799,62.6
salt_100g,22922,32.7
sodium_100g,22922,32.7
brands,19206,27.4
saturated_fat_100g,18800,26.9
sugars_100g,18129,25.9
fat_100g,16039,22.9


**Heads-up:** `nutriscore_grade` will look almost complete here, because Open
Food Facts stores a missing grade as the *text* `"unknown"` / `"not-applicable"`,
which `isna()` doesn't count. It should be fix that in step 5, and the real missing count
jumps up then — that's correct, not a regression.

In [5]:
## 4. Remove duplicate rows
before = len(df)
df = df.drop_duplicates(subset=["code", "country"]).reset_index(drop=True)
print(f"Removed {before - len(df)} duplicate (code, country) rows. Now {len(df)} rows.")

Removed 0 duplicate (code, country) rows. Now 70000 rows.


In [6]:
## 5. Clean the text fields
#Trim whitespace, turn blanks into real missing values, standardise the grades,
#and convert Open Food Facts' `"unknown"` placeholder into proper `NaN`.

for col in ["product_name", "brands", "nutriscore_grade", "eco_grade"]:
    df[col] = df[col].astype("string").str.strip()
    df[col] = df[col].replace({"": pd.NA})

# Grades: A-E uppercase; "unknown"/"not-applicable" -> missing
for col in ["nutriscore_grade", "eco_grade"]:
    df[col] = df[col].str.upper().replace({"UNKNOWN": pd.NA, "NOT-APPLICABLE": pd.NA})

# A product with no name is useless -> drop it
before = len(df)
df = df[df["product_name"].notna()].reset_index(drop=True)
print(f"Dropped {before - len(df)} rows with no product name. Now {len(df)} rows.")

Dropped 0 rows with no product name. Now 70000 rows.


In [7]:
## 6. Fix data types & remove impossible values

#Per-100g values must sit in a sensible range. Anything outside it is a data-entry
#error, so we blank out *that value* (not the whole product).

num_cols = ["nutriscore_score", "nova_group", "additives_n", "energy_kcal_100g",
            "sugars_100g", "fat_100g", "saturated_fat_100g", "salt_100g",
            "sodium_100g", "proteins_100g", "fiber_100g", "completeness"]

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

gram_cols = ["sugars_100g", "fat_100g", "saturated_fat_100g",
             "salt_100g", "sodium_100g", "proteins_100g", "fiber_100g"]

flagged = 0
for col in gram_cols:                       # grams per 100g must be 0–100
    bad = (df[col] < 0) | (df[col] > 100)
    flagged += int(bad.sum())
    df.loc[bad, col] = np.nan

bad_energy = (df["energy_kcal_100g"] < 0) | (df["energy_kcal_100g"] > 900)
flagged += int(bad_energy.sum())
df.loc[bad_energy, "energy_kcal_100g"] = np.nan

df.loc[~df["nova_group"].isin([1, 2, 3, 4]), "nova_group"] = np.nan

print(f"Blanked out {flagged} impossible nutrient values.")

Blanked out 119 impossible nutrient values.


In [8]:
## 7. Wrangling — derive helper columns
# Salt and sodium are linked (salt ≈ sodium × 2.5) — fill one from the other.

df["salt_100g"]   = df["salt_100g"].fillna(df["sodium_100g"] * 2.5)
df["sodium_100g"] = df["sodium_100g"].fillna(df["salt_100g"] / 2.5)

# Flag: does this product have the core nutrients the model needs?
core = ["energy_kcal_100g", "sugars_100g", "fat_100g",
        "saturated_fat_100g", "salt_100g", "proteins_100g"]
df["has_full_nutrition"] = df[core].notna().all(axis=1)

df["last_modified"] = pd.to_datetime(df["last_modified"], errors="coerce", utc=True)
df["last_modified_year"] = df["last_modified"].dt.year
print("Products with full core nutrition:",
      f"{df['has_full_nutrition'].mean()*100:.1f}%")

Products with full core nutrition: 63.8%


In [12]:
## 8. Missing-value strategy — handle each column the *right* way

#Not everything should be filled. Filling a numeric measurement with a mean or 0
##would bias every dashboard average **and** lower XGBoost's accuracy (it handles
#`NaN`). So we only fill where a missing value is itself a
#meaningful label, and we deliberately keep the measurement gaps.
# --- FILL: label columns where "missing" is itself information ---

df["brands"]    = df["brands"].fillna("Unknown")
df["eco_grade"] = df["eco_grade"].fillna("Unknown")

# --- KEEP AS NaN (on purpose): ---
#   * nutrient measurements -> filling would bias averages and hurt XGBoost
#   * additives_n           -> missing means "not analysed", NOT zero
#   * nova_group            -> nulls become their own group in Tableau
#   * nutriscore_grade      -> the TARGET; missing rows are dropped later,
#                              in notebook 04 only (dashboards keep them)

# Optional: a display-friendly grade so dashboards can show an "Unknown" bar
# without affecting the model (which uses the real nutriscore_grade column).

df["grade_display"] = df["nutriscore_grade"].fillna("Unknown")

print("Missing % after handling (gaps that remain are kept by design):")
print((df.isna().mean()*100).round(1).sort_values(ascending=False))


Missing % after handling (gaps that remain are kept by design):
nova_group            73.0
additives_n           70.5
nutriscore_grade      66.2
nutriscore_score      66.2
fiber_100g            62.6
salt_100g             32.8
sodium_100g           32.8
saturated_fat_100g    26.9
sugars_100g           25.9
fat_100g              22.9
proteins_100g         22.8
energy_kcal_100g      22.4
last_modified_year     0.0
has_full_nutrition     0.0
last_modified          0.0
completeness           0.0
code                   0.0
product_name           0.0
eco_grade              0.0
country_tag            0.0
country                0.0
brands                 0.0
grade_display          0.0
dtype: float64


In [11]:

## 9. Final check and save
print("Final shape:", df.shape)
assert len(df) > 0, "No rows left — check the steps above!"

print("\nRows per country:")
print(df["country"].value_counts().to_string())

print("\nTrainable rows (known A–E grade):", df["nutriscore_grade"].notna().sum())
print("Grade distribution:")
print(df["grade_display"].value_counts().to_string())

df.to_csv(CLEAN_PATH, index=False)
print(f"\nSaved clean data -> {CLEAN_PATH.resolve()}")


Final shape: (70000, 23)

Rows per country:
country
Belgium           7000
France            7000
Germany           7000
Italy             7000
Netherlands       7000
Poland            7000
Spain             7000
Sweden            7000
Switzerland       7000
United Kingdom    7000

Trainable rows (known A–E grade): 23675
Grade distribution:
grade_display
Unknown    46325
D           5954
E           5838
C           5471
A           3646
B           2766

Saved clean data -> /Users/kanakyadav/Documents/GitHub/capstone/data/products_clean.csv


## What I did

1. Audited missing values (and noted the `"unknown"` string trap)
2. Dropped duplicate (product, country) rows and nameless products
3. Standardised grades to A–E; converted `"unknown"` → `NaN`
4. Forced correct dtypes and blanked impossible nutrient values
5. Reconciled salt↔sodium; added `has_full_nutrition`
6. **Handled missing values per column type**: filled label columns
   (`brands`, `eco_grade`) with `"Unknown"`, kept nutrient gaps as `NaN`
   on purpose, and added a dashboard-friendly `grade_display`

**Next:** `03_feature_engineering.ipynb` reads `products_clean.csv`. We kept the
nutrient `NaN`s precisely because XGBoost wants them — imputation decisions (if
any) belong there, not here.